# Neighbourhood correlation metrics — cross-model evaluation

Per-cell marker-gene correlation with KNN neighbours, stratified by `library_key` / `dataset_key` / `technical_covariate_keys`. Uses purpose-based covariate keys (no `batch_key`). Outputs per-model and cross-model headline metrics (H1-H14), composite scores, distribution overlaps, leaf-level failure-mode assignments, and summary figures.

**Immune run (default params)**:
```
papermill docs/notebooks/model_comparisons/neighbourhood_correlation_metrics.ipynb \
          docs/notebooks/model_comparisons/neighbourhood_correlation_metrics_out.ipynb
```

**Bone-marrow (single-dataset) run**:
```
papermill docs/notebooks/model_comparisons/neighbourhood_correlation_metrics.ipynb \
          docs/notebooks/model_comparisons/neighbourhood_correlation_metrics_bm_out.ipynb \
          -p dataset_key '' -p models_filter bm_mm_ \
          -p output_dir docs/notebooks/model_comparisons/results_neighbourhood_correlation_bm/
```

**LSF submission** (100 GB holds dense markers matrix + connectivities + per-model parquet outputs):
```
bsub -q normal -n 8 -M 100000 -R"select[mem>100000] rusage[mem=100000] span[hosts=1]" \
     -e ./%J.err -o ./%J.out -J nbhd_corr \
     'PYTHONNOUSERSITE=TRUE module load ISG/conda && conda activate regularizedvi && \
      papermill docs/notebooks/model_comparisons/neighbourhood_correlation_metrics.ipynb \
                docs/notebooks/model_comparisons/neighbourhood_correlation_metrics_out.ipynb'
```

In [ ]:
adata_path = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration/adata_rna.h5ad"
results_base = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results"
models_tsv = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/docs/notebooks/model_comparisons/integration_metrics_v2.tsv"
models_filter = ""
library_key = "batch"
dataset_key = "dataset"
technical_covariate_keys = "is_tea_seq"
marker_csv = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/docs/notebooks/known_marker_genes.csv"
label_columns = "level_1,level_2,level_3,level_4,harmonized_annotation"
output_dir = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/docs/notebooks/model_comparisons/results_neighbourhood_correlation/"
k_reference = 50
mean_threshold = 1.0
specificity_threshold = 0.1
top_n_per_label = 500
tea_seq_dataset_value = "pbmc_tea_seq"
random_state = 42

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt

from regularizedvi.plt._neighbourhood_correlation import (
    select_marker_genes,
    validate_covariate_hierarchy,
    construct_neighbour_masks,
    list_active_masks,
    compute_marker_correlation,
    compute_neighbourhood_diagnostics,
    compute_random_knn_baseline,
    compute_analytical_isolation_baseline,
    classify_failure_modes,
    summarize_failure_modes,
    compute_distribution_overlap,
    compute_isolation_norm,
    summarise_marker_correlation,
    compute_composite_score,
    assemble_cross_model_metrics,
    compute_best_achievable,
    compute_integration_failure_rate,
    compute_model_pair_overlaps,
    compute_cross_technical_correlation,
    flag_consensus_isolated,
    plot_marker_correlation_umap,
    plot_metric_hist2d,
    plot_failure_mode_scatter,
    plot_distribution_overlap,
    plot_per_library_distributions,
    plot_isolation_bars,
    plot_leaf_distribution,
)
from regularizedvi.utils import coerce_papermill_params

_coerced = coerce_papermill_params(
    k_reference=(k_reference, int),
    mean_threshold=(mean_threshold, float),
    specificity_threshold=(specificity_threshold, float),
    top_n_per_label=(top_n_per_label, int),
    random_state=(random_state, int),
)
k_reference = _coerced["k_reference"]
mean_threshold = _coerced["mean_threshold"]
specificity_threshold = _coerced["specificity_threshold"]
top_n_per_label = _coerced["top_n_per_label"]
random_state = _coerced["random_state"]

Path(output_dir).mkdir(parents=True, exist_ok=True)


def _split(s):
    if s is None or s == "":
        return []
    if isinstance(s, list):
        return list(s)
    return [x.strip() for x in s.split(",") if x.strip()]


label_columns_list = _split(label_columns)
technical_covariate_keys_list = _split(technical_covariate_keys) or None
dataset_key_eff = dataset_key if dataset_key else None

print(f"adata:          {adata_path}")
print(f"library_key:    {library_key}")
print(f"dataset_key:    {dataset_key_eff}")
print(f"technical:      {technical_covariate_keys_list}")
print(f"models_tsv:     {models_tsv}")
print(f"output_dir:     {output_dir}")
print(f"top_n_per_label:{top_n_per_label}")

In [ ]:
adata = sc.read_h5ad(adata_path)
print(adata)

if dataset_key_eff is not None and "is_tea_seq" in (technical_covariate_keys_list or []):
    adata.obs["is_tea_seq"] = (
        adata.obs[dataset_key_eff] == tea_seq_dataset_value
    ).astype("category")
    n_tea = int((adata.obs["is_tea_seq"].astype(str) == "True").sum())
    print(f"is_tea_seq: {n_tea} TEA-seq cells, {adata.n_obs - n_tea} other")

validate_covariate_hierarchy(adata, library_key=library_key, dataset_key=dataset_key_eff)

In [ ]:
if not Path(models_tsv).exists():
    raise FileNotFoundError(f"models_tsv not found: {models_tsv}")

registry = pd.read_csv(models_tsv, sep="\t").drop_duplicates(subset=["name"])

if models_filter:
    registry = registry[registry["name"].str.contains(models_filter, regex=False)]

print(f"Registry: {len(registry)} candidate models")
if "type" in registry.columns:
    print(registry["type"].value_counts().to_dict())
registry[["name"]].head(40)

## 2. Gene selection
Curated markers + data-driven selection per dataset.

In [ ]:
curated = pd.read_csv(marker_csv)
print(f"Curated marker genes: {len(curated)} rows, columns {list(curated.columns)}")

selected = select_marker_genes(
    adata,
    label_columns=label_columns_list,
    curated_marker_csv=marker_csv,
    mean_threshold=mean_threshold,
    specificity_threshold=specificity_threshold,
    top_n_per_label=top_n_per_label,
    per_dataset=dataset_key_eff is not None,
    dataset_col=dataset_key_eff if dataset_key_eff else "dataset",
)

marker_genes = selected["union"]
print(f"Total selected marker genes (union): {len(marker_genes)}")
print(f"  curated:          {len(selected['curated'])}")
print(f"  data-driven:      {len(selected['data_driven'])}")
print(f"  top_n_per_label:  {top_n_per_label}")

## 3. Per-model neighbourhood correlation pipeline
For each model with a saved connectivities matrix, compute diagnostics, marker correlation, random baseline, failure-mode leaves. Persist per-cell outputs to parquet.

In [ ]:
import time

pca_conn = None
if "pca_baseline" in registry["name"].values:
    t0 = time.time()
    print("Computing PCA baseline on log1p(raw counts)...")
    adata_pca = adata.copy()
    sc.pp.log1p(adata_pca)
    sc.pp.pca(adata_pca, n_comps=100, svd_solver="arpack")
    X_pca = adata_pca.obsm["X_pca"]
    total_counts = np.asarray(adata.X.sum(axis=1)).ravel()
    pc1_corr = float(np.corrcoef(X_pca[:, 0], total_counts)[0, 1])
    print(f"PC1 correlation with total counts: {pc1_corr:.4f}")
    if abs(pc1_corr) > 0.5:
        print(f"Removing PC1 (high correlation: {pc1_corr:.4f})")
        X_pca = X_pca[:, 1:]
    adata.obsm["X_pca_baseline"] = X_pca

    print(f"Building PCA KNN graph k={k_reference}...")
    sc.pp.neighbors(
        adata, use_rep="X_pca_baseline",
        n_neighbors=k_reference, key_added="pca",
    )
    pca_conn = adata.obsp["pca_connectivities"].tocsr()
    del adata_pca
    print(f"PCA baseline ready: {X_pca.shape} in {time.time() - t0:.1f}s")

In [ ]:
from regularizedvi.plt._neighbourhood_correlation import normalise_counts

per_model_metrics = {}
per_model_leaves = {}
per_model_diagnostics = {}
per_model_baseline = {}
skipped = []

# Pre-compute normalised markers matrix once (model-independent)
marker_mask = adata.var_names.isin(marker_genes)
X_full = adata.X
total_counts_full = np.asarray(X_full.sum(axis=1)).flatten()
X_markers = adata[:, marker_mask].X
if not sp.issparse(X_markers):
    X_markers = sp.csr_matrix(X_markers)
X_markers = X_markers.tocsr().astype(np.float32)
X_markers_norm = normalise_counts(
    X_markers, n_vars=adata.n_vars, total_counts=total_counts_full,
).astype(np.float32)

for _, row in registry.iterrows():
    model_name = row["name"]

    if model_name == "pca_baseline":
        if pca_conn is None:
            skipped.append(model_name)
            continue
        conn = pca_conn
    else:
        conn_path = Path(results_base) / model_name / "model" / "outputs" / f"connectivities_euclidean_k{k_reference}.npz"
        if not conn_path.exists():
            skipped.append(model_name)
            continue
        conn = sp.load_npz(conn_path).tocsr()

    if conn.shape[0] != adata.n_obs:
        print(f"  {model_name}: connectivity shape {conn.shape} != adata.n_obs {adata.n_obs}; skipping")
        skipped.append(model_name)
        continue

    diag = compute_neighbourhood_diagnostics(
        adata, conn,
        library_key=library_key,
        dataset_key=dataset_key_eff,
        technical_covariate_keys=technical_covariate_keys_list,
    )

    metrics = compute_marker_correlation(
        adata, conn, marker_genes,
        library_key=library_key,
        dataset_key=dataset_key_eff,
        technical_covariate_keys=technical_covariate_keys_list,
    )

    degree_per_cell = np.asarray(
        (conn.copy().astype(bool)).sum(axis=1)
    ).flatten().astype(np.int64)

    baseline = compute_random_knn_baseline(
        adata, X_markers_norm, degree_per_cell,
        library_key=library_key,
        dataset_key=dataset_key_eff,
        random_state=random_state,
    )

    leaves = classify_failure_modes(
        metrics, random_baseline_df=baseline,
    )

    per_model_metrics[model_name] = metrics
    per_model_leaves[model_name] = leaves
    per_model_diagnostics[model_name] = diag
    per_model_baseline[model_name] = baseline

    metrics.to_parquet(f"{output_dir}/{model_name}_metrics.parquet")
    leaves.to_parquet(f"{output_dir}/{model_name}_leaves.parquet")
    print(f"  {model_name}: done")

print(f"\nEvaluated models: {len(per_model_metrics)}; skipped: {len(skipped)}")
if skipped:
    print(f"Skipped (no connectivity file): {skipped[:10]}{' ...' if len(skipped) > 10 else ''}")

## 4. Per-model headline + composite scores

In [ ]:
headline_rows = {}
composite_rows = {}
for model_name, metrics in per_model_metrics.items():
    headline = summarise_marker_correlation(
        metrics,
        adata=adata,
        library_key=library_key,
        dataset_key=dataset_key_eff,
        technical_covariate_keys=technical_covariate_keys_list,
        random_baseline_df=per_model_baseline.get(model_name),
    )
    headline_rows[model_name] = headline
    composite_rows[model_name] = compute_composite_score(
        headline,
        has_dataset=dataset_key_eff is not None,
    )

per_model_headline = pd.DataFrame(headline_rows).T
composite = pd.DataFrame(composite_rows).T

per_model_headline.to_csv(f"{output_dir}/per_model_headline.csv")
composite.sort_values("total", ascending=False).to_csv(f"{output_dir}/composite_scores.csv")

display(composite.sort_values("total", ascending=False).round(3))

## 5. Cross-model comparison

In [ ]:
cross_model_df = assemble_cross_model_metrics(per_model_metrics)

# H13 integration_failure_rate per model (cells achievable by any model but this one fails)
ifr = {m: compute_integration_failure_rate(cross_model_df, m) for m in per_model_metrics}
per_model_headline["integration_failure_rate"] = pd.Series(ifr)
per_model_headline.to_csv(f"{output_dir}/per_model_headline.csv")

# Pairwise OVL on corr_avg_cross_dataset (if available)
overlap_metric = "corr_avg_cross_dataset" if dataset_key_eff else "corr_avg_same_library"
ovl = compute_model_pair_overlaps(cross_model_df, metric_name=overlap_metric)
ovl.to_csv(f"{output_dir}/cross_model_overlap.csv")

# Consensus-isolated cells (cross-dataset only when dataset_key set)
if dataset_key_eff is not None:
    consensus = flag_consensus_isolated(cross_model_df)
    consensus.to_frame("consensus_isolated").to_csv(f"{output_dir}/consensus_isolated_cells.csv")
    print(f"Consensus-isolated cells: {int(consensus.sum())} / {len(consensus)}")

display(
    per_model_headline[
        [
            "corr_within_library",
            "corr_cross_library",
            "corr_cross_dataset",
            "integration_failure_rate",
            "cross_technical_correlation",
        ]
    ].round(3)
)

## 6. Visualisations (V1-V14 from main plan)
One figure per visualisation, saved under `output_dir`.

In [ ]:
import matplotlib

matplotlib.use("Agg")

has_umap = "X_umap" in adata.obsm
dataset_key_for_plots = dataset_key_eff if dataset_key_eff is not None else library_key

# V1/V12: UMAP colour by corr_avg_same_library per model
if has_umap:
    for model_name, metrics in per_model_metrics.items():
        fig = plot_marker_correlation_umap(
            adata,
            metrics,
            columns=["corr_avg_same_library"],
        )
        fig.savefig(f"{output_dir}/umap_{model_name}_corr_sl.png", dpi=120, bbox_inches="tight")
        plt.close(fig)

# V2-V6/V14: hist2d per model (same_library vs cross_library)
for model_name, metrics in per_model_metrics.items():
    if "corr_avg_cross_library" not in metrics.columns:
        continue
    fig = plot_metric_hist2d(
        metrics,
        x_col="corr_avg_same_library",
        y_col="corr_avg_cross_library",
    )
    fig.savefig(f"{output_dir}/hist2d_{model_name}_sl_vs_xl.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

# V7/V8: distribution overlap per model (same_library vs cross_library)
for model_name, metrics in per_model_metrics.items():
    if "corr_avg_cross_library" not in metrics.columns:
        continue
    fig = plot_distribution_overlap(
        metrics,
        col_a="corr_avg_same_library",
        col_b="corr_avg_cross_library",
    )
    fig.savefig(f"{output_dir}/dist_overlap_{model_name}_library.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

# V9: per-library distributions (one row per dataset) — require dataset_key column
for model_name, metrics in per_model_metrics.items():
    if dataset_key_for_plots not in adata.obs.columns:
        continue
    fig = plot_per_library_distributions(
        metrics,
        adata,
        library_key=library_key,
        dataset_key=dataset_key_for_plots,
    )
    fig.savefig(f"{output_dir}/per_library_{model_name}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

# V10: isolation bars across models (expects {model: headline_series})
fig = plot_isolation_bars(headline_rows)
fig.savefig(f"{output_dir}/isolation_bars.png", dpi=120, bbox_inches="tight")
plt.close(fig)

# V13: leaf distribution across models (expects {model: failure_mode_series})
per_model_failure_mode = {
    m: leaves["failure_mode"] if "failure_mode" in leaves.columns else leaves.iloc[:, 0]
    for m, leaves in per_model_leaves.items()
}
fig = plot_leaf_distribution(per_model_failure_mode)
fig.savefig(f"{output_dir}/leaves_failure_mode.png", dpi=120, bbox_inches="tight")
plt.close(fig)

print("Figures saved to", output_dir)

## 7. TEA-seq case study (dataset-specific failure mode)
Compare `scvi_baseline` vs `regularizedVI` restricted to TEA-seq cells. Expect high `integration_failure_rate` for scvi_baseline.

In [ ]:
if dataset_key_eff is None or tea_seq_dataset_value not in adata.obs[dataset_key_eff].unique():
    print("TEA-seq case study skipped: dataset_key or pbmc_tea_seq value not present in adata")
else:
    tea_mask = (adata.obs[dataset_key_eff] == tea_seq_dataset_value).to_numpy()
    tea_index = adata.obs_names[tea_mask]

    tea_headline_rows = {}
    for model_name, metrics in per_model_metrics.items():
        sub = metrics.loc[metrics.index.intersection(tea_index)]
        if sub.empty:
            continue
        tea_headline_rows[model_name] = summarise_marker_correlation(
            sub,
            adata=adata[sub.index].copy(),
            library_key=library_key,
            dataset_key=dataset_key_eff,
            technical_covariate_keys=technical_covariate_keys_list,
        )
    tea_headline = pd.DataFrame(tea_headline_rows).T

    # Failure rate restricted to TEA-seq
    tea_cross_model = cross_model_df.loc[cross_model_df.index.intersection(tea_index)]
    tea_ifr = {m: compute_integration_failure_rate(tea_cross_model, m) for m in per_model_metrics}
    tea_headline["integration_failure_rate_tea"] = pd.Series(tea_ifr)

    tea_headline.to_csv(f"{output_dir}/tea_seq_case_study.csv")
    display(tea_headline.round(3))

## 8. Embryo / other datasets (template)
Override papermill parameters for different datasets. Example for embryo:
```python
library_key = "sample_id"
dataset_key = "Section"
technical_covariate_keys = "Embryo,Experiment"
```
For bone-marrow single-dataset runs set `dataset_key=""` (empty string) and pass a `models_filter` matching the BM experiment name prefix.

In [ ]:
print("Done.")
print("Outputs:")
for p in sorted(Path(output_dir).glob("*"))[:20]:
    print(" ", p)